# 05 - Transfer learning

En esta notebook se prueba el bonus de la Parte 2: transfer learning.

Se utiliza ResNet18 preentrenada en ImageNet y se comparan distintas estrategias:

- entrenar solo la capa final;
- entrenar el último bloque convolucional + capa final;
- comparar augmentations.

El test fijo no se usa en esta notebook. La selección se realiza con validación sobre `trainval`.

In [1]:
from pathlib import Path
import os
import sys
import json

import pandas as pd

REPO_ROOT = Path.cwd()

if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)

src_path = REPO_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from transfer_learning_training import run_transfer_cross_validation
from cnn_training import save_experiment_outputs

print("Repo root:", REPO_ROOT)

Repo root: c:\Users\tomas\OneDrive\Documentos\MATERIAS ITBA\ELECTIVAS - CUATRIMESTRE X\MACHINE LEARNING y REDES NEURONALES EN BIOINGENIERÍA\skin-dataset-classification


In [2]:
base_transfer_config = {
    "mlflow_experiment": "TP_skin_transfer_learning",

    "image_size": 224,
    "batch_size": 16,
    "epochs": 8,
    "lr": 3e-4,
    "optimizer": "adamw",

    "dropout": 0.3,
    "weight_decay": 1e-4,
    "augmentation": "minimal",

    "strategy": "freeze",

    "early_stopping": True,
    "patience": 4,

    "tensorboard": True,
    "mlflow": True,
    "log_pytorch_model": False,

    "seed": 42,
}

In [3]:
quick_transfer_config = base_transfer_config.copy()
quick_transfer_config["experiment"] = "TL_QUICK_CHECK"
quick_transfer_config["epochs"] = 2
quick_transfer_config["log_pytorch_model"] = False

quick_transfer_output = run_transfer_cross_validation(
    config=quick_transfer_config,
    split_csv="data/splits/final_split_5fold.csv",
    folds_to_run=[0],
)

display(quick_transfer_output["fold_results"])
quick_transfer_output["summary"]

Device usado: cpu
Folds a correr: [0]
Clases: {0: 'Actinic keratosis', 1: 'Atopic Dermatitis', 2: 'Benign keratosis', 3: 'Dermatofibroma', 4: 'Melanocytic nevus', 5: 'Melanoma', 6: 'Squamous cell carcinoma', 7: 'Tinea Ringworm Candidiasis', 8: 'Vascular lesion'}
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\tomas/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:08<00:00, 5.59MB/s]
2026/06/18 12:57:42 INFO mlflow.tracking.fluent: Experiment with name 'TP_skin_transfer_learning' does not exist. Creating a new experiment.


Parámetros entrenables: 4,617
[TL_QUICK_CHECK] fold 0 | epoch 01/2 | train_acc=0.164 | val_acc=0.257 | val_f1=0.211
[TL_QUICK_CHECK] fold 0 | epoch 02/2 | train_acc=0.241 | val_acc=0.375 | val_f1=0.347


,experiment,fold,best_epoch,val_loss,val_accuracy,val_macro_f1,val_balanced_accuracy,n_train,n_val
0,TL_QUICK_CHECK,0,2,1.803583,0.375,0.34682,0.358543,573,144


{'experiment': 'TL_QUICK_CHECK',
 'model': 'ResNet18_pretrained',
 'strategy': 'freeze',
 'image_size': 224,
 'batch_size': 16,
 'epochs': 2,
 'lr': 0.0003,
 'optimizer': 'adamw',
 'dropout': 0.3,
 'weight_decay': 0.0001,
 'augmentation': 'minimal',
 'early_stopping': True,
 'folds_run': [0],
 'best_fold': 0,
 'num_trainable_params': 4617,
 'val_accuracy_mean': 0.375,
 'val_accuracy_std': 0.0,
 'val_macro_f1_mean': 0.34682032169298727,
 'val_macro_f1_std': 0.0,
 'val_balanced_accuracy_mean': 0.3585434173669468,
 'val_balanced_accuracy_std': 0.0,
 'mlflow_run_id': '8d855ada965042ffbc5419f65dab98d9'}

In [4]:
save_experiment_outputs(
    cv_output=quick_transfer_output,
    output_prefix="tl_quick_check",
)

Archivos guardados:
- experiments\tl_quick_check_fold_results.csv
- experiments\tl_quick_check_history.csv
- experiments\tl_quick_check_summary.json
- results\training_curves\tl_quick_check_loss.png
- results\training_curves\tl_quick_check_accuracy.png
- results\confusion_matrices\tl_quick_check.png
- results\classification_reports\tl_quick_check_classification_report.txt


In [5]:
transfer_screening_configs = []

def make_transfer_config(name, **updates):
    cfg = base_transfer_config.copy()
    cfg["experiment"] = name
    cfg.update(updates)
    return cfg

transfer_screening_configs.append(
    make_transfer_config(
        "TL_SCREEN_1_resnet18_freeze_minimal",
        strategy="freeze",
        augmentation="minimal",
        lr=3e-4,
    )
)

transfer_screening_configs.append(
    make_transfer_config(
        "TL_SCREEN_2_resnet18_freeze_light",
        strategy="freeze",
        augmentation="light",
        lr=3e-4,
    )
)

transfer_screening_configs.append(
    make_transfer_config(
        "TL_SCREEN_3_resnet18_layer4_minimal",
        strategy="fine_tune_last_block",
        augmentation="minimal",
        lr=1e-4,
    )
)

len(transfer_screening_configs), [cfg["experiment"] for cfg in transfer_screening_configs]

(3,
 ['TL_SCREEN_1_resnet18_freeze_minimal',
  'TL_SCREEN_2_resnet18_freeze_light',
  'TL_SCREEN_3_resnet18_layer4_minimal'])

In [6]:
transfer_screening_summaries = []

for cfg in transfer_screening_configs:
    print("\n" + "=" * 80)
    print("Corriendo:", cfg["experiment"])
    print("=" * 80)

    output = run_transfer_cross_validation(
        config=cfg,
        split_csv="data/splits/final_split_5fold.csv",
        folds_to_run=[0],
    )

    output_prefix = cfg["experiment"].lower()
    save_experiment_outputs(
        cv_output=output,
        output_prefix=output_prefix,
    )

    transfer_screening_summaries.append(output["summary"])

transfer_screening_df = pd.DataFrame(transfer_screening_summaries)
transfer_screening_df = transfer_screening_df.sort_values("val_accuracy_mean", ascending=False)

display(transfer_screening_df[[
    "experiment",
    "model",
    "strategy",
    "dropout",
    "weight_decay",
    "lr",
    "augmentation",
    "num_trainable_params",
    "val_accuracy_mean",
    "val_macro_f1_mean",
    "val_balanced_accuracy_mean",
]])


Corriendo: TL_SCREEN_1_resnet18_freeze_minimal
Device usado: cpu
Folds a correr: [0]
Clases: {0: 'Actinic keratosis', 1: 'Atopic Dermatitis', 2: 'Benign keratosis', 3: 'Dermatofibroma', 4: 'Melanocytic nevus', 5: 'Melanoma', 6: 'Squamous cell carcinoma', 7: 'Tinea Ringworm Candidiasis', 8: 'Vascular lesion'}
Parámetros entrenables: 4,617
[TL_SCREEN_1_resnet18_freeze_minimal] fold 0 | epoch 01/8 | train_acc=0.164 | val_acc=0.257 | val_f1=0.211
[TL_SCREEN_1_resnet18_freeze_minimal] fold 0 | epoch 02/8 | train_acc=0.241 | val_acc=0.375 | val_f1=0.347
[TL_SCREEN_1_resnet18_freeze_minimal] fold 0 | epoch 03/8 | train_acc=0.328 | val_acc=0.451 | val_f1=0.433
[TL_SCREEN_1_resnet18_freeze_minimal] fold 0 | epoch 04/8 | train_acc=0.449 | val_acc=0.569 | val_f1=0.546
[TL_SCREEN_1_resnet18_freeze_minimal] fold 0 | epoch 05/8 | train_acc=0.497 | val_acc=0.632 | val_f1=0.630
[TL_SCREEN_1_resnet18_freeze_minimal] fold 0 | epoch 06/8 | train_acc=0.515 | val_acc=0.618 | val_f1=0.611
[TL_SCREEN_1_resn

,experiment,model,strategy,dropout,weight_decay,lr,augmentation,num_trainable_params,val_accuracy_mean,val_macro_f1_mean,val_balanced_accuracy_mean
2,TL_SCREEN_3_resnet18_layer4_minimal,ResNet18_pretrained,fine_tune_last_block,0.3,0.0001,0.0001,minimal,8398345,0.847222,0.855461,0.856209
0,TL_SCREEN_1_resnet18_freeze_minimal,ResNet18_pretrained,freeze,0.3,0.0001,0.0003,minimal,4617,0.673611,0.668063,0.670020
1,TL_SCREEN_2_resnet18_freeze_light,ResNet18_pretrained,freeze,0.3,0.0001,0.0003,light,4617,0.631944,0.630235,0.632204


In [7]:
transfer_screening_path = Path("experiments/transfer_learning_screening_summary.csv")
transfer_screening_df.to_csv(transfer_screening_path, index=False)
print("Guardado:", transfer_screening_path)

Guardado: experiments\transfer_learning_screening_summary.csv


In [8]:
best_transfer_name = transfer_screening_df.iloc[0]["experiment"]
best_transfer_name

best_transfer_config = [
    cfg for cfg in transfer_screening_configs
    if cfg["experiment"] == best_transfer_name
][0].copy()

best_transfer_config["experiment"] = best_transfer_config["experiment"].replace("TL_SCREEN_", "TL_FULL_")
best_transfer_config["epochs"] = 15
best_transfer_config["patience"] = 5
best_transfer_config["log_pytorch_model"] = True

best_transfer_config

{'mlflow_experiment': 'TP_skin_transfer_learning',
 'image_size': 224,
 'batch_size': 16,
 'epochs': 15,
 'lr': 0.0001,
 'optimizer': 'adamw',
 'dropout': 0.3,
 'weight_decay': 0.0001,
 'augmentation': 'minimal',
 'strategy': 'fine_tune_last_block',
 'early_stopping': True,
 'patience': 5,
 'tensorboard': True,
 'mlflow': True,
 'log_pytorch_model': True,
 'seed': 42,
 'experiment': 'TL_FULL_3_resnet18_layer4_minimal'}

In [9]:
transfer_full_output = run_transfer_cross_validation(
    config=best_transfer_config,
    split_csv="data/splits/final_split_5fold.csv",
    folds_to_run=None,
)

display(transfer_full_output["fold_results"])
transfer_full_output["summary"]

Device usado: cpu
Folds a correr: [0, 1, 2, 3, 4]
Clases: {0: 'Actinic keratosis', 1: 'Atopic Dermatitis', 2: 'Benign keratosis', 3: 'Dermatofibroma', 4: 'Melanocytic nevus', 5: 'Melanoma', 6: 'Squamous cell carcinoma', 7: 'Tinea Ringworm Candidiasis', 8: 'Vascular lesion'}
Parámetros entrenables: 8,398,345
[TL_FULL_3_resnet18_layer4_minimal] fold 0 | epoch 01/15 | train_acc=0.497 | val_acc=0.806 | val_f1=0.807
[TL_FULL_3_resnet18_layer4_minimal] fold 0 | epoch 02/15 | train_acc=0.860 | val_acc=0.819 | val_f1=0.826
[TL_FULL_3_resnet18_layer4_minimal] fold 0 | epoch 03/15 | train_acc=0.927 | val_acc=0.812 | val_f1=0.817
[TL_FULL_3_resnet18_layer4_minimal] fold 0 | epoch 04/15 | train_acc=0.983 | val_acc=0.799 | val_f1=0.805
[TL_FULL_3_resnet18_layer4_minimal] fold 0 | epoch 05/15 | train_acc=0.995 | val_acc=0.847 | val_f1=0.855
[TL_FULL_3_resnet18_layer4_minimal] fold 0 | epoch 06/15 | train_acc=0.993 | val_acc=0.806 | val_f1=0.810
[TL_FULL_3_resnet18_layer4_minimal] fold 0 | epoch 07/1

2026/06/18 14:29:52 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


,experiment,fold,best_epoch,val_loss,val_accuracy,val_macro_f1,val_balanced_accuracy,n_train,n_val
0,TL_FULL_3_resnet18_layer4_minimal,0,5,0.577224,0.847222,0.855461,0.856209,573,144
1,TL_FULL_3_resnet18_layer4_minimal,1,3,0.566537,0.840278,0.846221,0.846872,573,144
2,TL_FULL_3_resnet18_layer4_minimal,2,6,0.565845,0.804196,0.817628,0.814192,574,143
3,TL_FULL_3_resnet18_layer4_minimal,3,12,0.567550,0.832168,0.835264,0.837561,574,143
4,TL_FULL_3_resnet18_layer4_minimal,4,5,0.550189,0.825175,0.830963,0.832579,574,143


{'experiment': 'TL_FULL_3_resnet18_layer4_minimal',
 'model': 'ResNet18_pretrained',
 'strategy': 'fine_tune_last_block',
 'image_size': 224,
 'batch_size': 16,
 'epochs': 15,
 'lr': 0.0001,
 'optimizer': 'adamw',
 'dropout': 0.3,
 'weight_decay': 0.0001,
 'augmentation': 'minimal',
 'early_stopping': True,
 'folds_run': [0, 1, 2, 3, 4],
 'best_fold': 0,
 'num_trainable_params': 8398345,
 'val_accuracy_mean': 0.8298076923076924,
 'val_accuracy_std': 0.014804668672962934,
 'val_macro_f1_mean': 0.8371074940553678,
 'val_macro_f1_std': 0.012965535316299985,
 'val_balanced_accuracy_mean': 0.8374827786592492,
 'val_balanced_accuracy_std': 0.014179786642273078,
 'mlflow_run_id': '9d5fab92907c4d2c904a97507c6250c3'}

In [10]:
save_experiment_outputs(
    cv_output=transfer_full_output,
    output_prefix=best_transfer_config["experiment"].lower(),
)

Archivos guardados:
- experiments\tl_full_3_resnet18_layer4_minimal_fold_results.csv
- experiments\tl_full_3_resnet18_layer4_minimal_history.csv
- experiments\tl_full_3_resnet18_layer4_minimal_summary.json
- results\training_curves\tl_full_3_resnet18_layer4_minimal_loss.png
- results\training_curves\tl_full_3_resnet18_layer4_minimal_accuracy.png
- results\confusion_matrices\tl_full_3_resnet18_layer4_minimal.png
- results\classification_reports\tl_full_3_resnet18_layer4_minimal_classification_report.txt


In [11]:
cnn_comparison_path = Path("experiments/cnn_full_comparison.csv")
cnn_df = pd.read_csv(cnn_comparison_path)

best_cnn_row = cnn_df.sort_values("val_accuracy_mean", ascending=False).iloc[0].to_dict()

transfer_row = {
    "experiment": transfer_full_output["summary"]["experiment"],
    "model": transfer_full_output["summary"]["model"],
    "strategy": transfer_full_output["summary"]["strategy"],
    "augmentation": transfer_full_output["summary"]["augmentation"],
    "val_accuracy_mean": transfer_full_output["summary"]["val_accuracy_mean"],
    "val_accuracy_std": transfer_full_output["summary"]["val_accuracy_std"],
    "val_macro_f1_mean": transfer_full_output["summary"]["val_macro_f1_mean"],
    "val_balanced_accuracy_mean": transfer_full_output["summary"]["val_balanced_accuracy_mean"],
}

final_model_comparison = pd.DataFrame([
    {
        "experiment": best_cnn_row["experiment"],
        "model": best_cnn_row["model"],
        "strategy": "trained_from_scratch",
        "augmentation": best_cnn_row["augmentation"],
        "val_accuracy_mean": best_cnn_row["val_accuracy_mean"],
        "val_accuracy_std": best_cnn_row["val_accuracy_std"],
        "val_macro_f1_mean": best_cnn_row["val_macro_f1_mean"],
        "val_balanced_accuracy_mean": best_cnn_row["val_balanced_accuracy_mean"],
    },
    transfer_row,
])

final_model_comparison = final_model_comparison.sort_values(
    "val_accuracy_mean",
    ascending=False,
)

display(final_model_comparison)

,experiment,model,strategy,augmentation,val_accuracy_mean,val_accuracy_std,val_macro_f1_mean,val_balanced_accuracy_mean
1,TL_FULL_3_resnet18_layer4_minimal,ResNet18_pretrained,fine_tune_last_block,minimal,0.829808,0.014805,0.837107,0.837483
0,CNN_FULL_4_aug_minimal,AlexNetSmall,trained_from_scratch,minimal,0.679157,0.020665,0.673943,0.673236


In [12]:
final_comparison_path = Path("experiments/cnn_vs_transfer_learning_comparison.csv")
final_model_comparison.to_csv(final_comparison_path, index=False)
print("Guardado:", final_comparison_path)

Guardado: experiments\cnn_vs_transfer_learning_comparison.csv
